![Banner](../images/08_bayesian_kriging.png)


# Bayesian kriging



Ordinary kriging fits a variogram, treats it as known, and then predicts. The prediction variance it reports covers the unknown value only. It ignores the fact that the variogram's nugget, sill and range were themselves estimated from the same few data. With small or moderate samples that error can be large, so plug-in kriging tends to be over-confident: its intervals are too narrow.

Bayesian kriging treats the unknown model parameters as random variables. These are the trend coefficients β, the variance σ², and the correlation parameters θ (range, nugget ratio, and, where the model has one, a smoothness or a Box-Cox λ). You give them prior distributions, update the priors with the data, and predict by averaging over everything the data still allow:

$$
p\bigl(z(\mathbf{u}_0) \mid \mathbf{z}\bigr) = \int p\bigl(z(\mathbf{u}_0) \mid \mathbf{z}, \beta, \sigma^2, \theta\bigr)\, p(\beta, \sigma^2, \theta \mid \mathbf{z})\, d\beta\, d\sigma^2\, d\theta .
$$

The first factor is the ordinary kriging predictor for one fixed set of parameters. The second is the posterior of the parameters. The result is a predictive distribution, not a single number with a variance. It is usually wider than the kriging interval, and it is better calibrated, especially when data are few or the range is poorly determined.

This notebook is the map for Bayesian kriging in GeoLattice. It names each class and function the library provides, says which uncertainty that class carries, and points to the notebook that runs it on data.

Eight notebooks carry the uncertainty in the variogram into the prediction.

- **Bayesian kriging.** This notebook. The predictive integral, and which class carries which part of the uncertainty.
- **Empirical Bayesian Kriging.** Jura log cadmium. A simulate-and-refit ensemble, leave-one-out weights, a check against ordinary kriging, and exceedance maps.
- **Bayesian Kriging (`BayesianKriging`).** A grid posterior over range and nugget, priors, model choice, and the Student-t mixture.
- **Bayesian Generalized Linear Kriging (`BayesianGLMKriging`).** Poisson lung-cancer counts and binomial diabetes prevalence for 666 Atlantic counties, with the true surface known.
- **Bayesian Transformed Gaussian Kriging (`BayesianTransformedKriging`).** A Box-Cox λ on the grid, back-transformed draws, and Jura and Meuse metals.
- **Conjugate Bayesian Kriging (`ConjugateBayesianKriging`).** A fixed correlation, the Student-t predictive, `from_variogram`, and trend priors, on Jura log cadmium.
- **Bayesian Kriging in One Call (`bayesian_kriging`).** `bayesian_kriging` returns the mean, the standard deviation, and the quantiles for Jura log cadmium.
- **Correlation Functions (`correlation_function`).** The six correlation models, practical range, Matérn smoothness, and the stable family.

Library: `geolattice.geostatistics`


---

## Table of Contents

- [What this chapter covers](#what-this-chapter-covers)
1. [Notebooks in this chapter](#notebooks-in-this-chapter)
2. [Types of Bayesian kriging](#types-of-bayesian-kriging)
    - 2.1 [Conjugate Bayesian kriging (known correlation)](#conjugate-bayesian-kriging-known-correlation)
    - 2.2 [Bayesian kriging with a discrete posterior (model-based geostatistics)](#bayesian-kriging-with-a-discrete-posterior-model-based-geostatistics)
    - 2.3 [Bayesian transformed Gaussian kriging](#bayesian-transformed-gaussian-kriging)
    - 2.4 [Bayesian generalized kriging (spatial GLMMs)](#bayesian-generalized-kriging-spatial-glmms)
    - 2.5 [Empirical Bayesian kriging (EBK)](#empirical-bayesian-kriging-ebk)
    - 2.6 [The correlation function](#the-correlation-function)
3. [Bayesian kriging in GeoLattice](#bayesian-kriging-in-geolattice)
    - 3.1 [`correlation_function`](#correlation_function)
    - 3.2 [`EmpiricalBayesianKriging`](#empiricalbayesiankriging)
    - 3.3 [`ConjugateBayesianKriging`](#conjugatebayesiankriging)
    - 3.4 [`BayesianKriging`](#bayesiankriging)
    - 3.5 [`BayesianTransformedKriging`](#bayesiantransformedkriging)
    - 3.6 [`BayesianGLMKriging`](#bayesianglmkriging)
    - 3.7 [`bayesian_kriging`](#bayesian_kriging)
4. [Choosing among them](#choosing-among-them)




## Notebooks in this chapter

| Notebook | Object | What it covers |
|---|---|---|
| [05_08_00_introduction_bayesian_kriging.ipynb](05_08_00_introduction_bayesian_kriging.ipynb) **(this)** | All of them, side by side | The predictive integral and the GeoLattice calls |
| [05_08_01_empirical_bayesian_kriging.ipynb](05_08_01_empirical_bayesian_kriging.ipynb) | `EmpiricalBayesianKriging` | Jura log(Cd): simulate-and-refit ensemble, LOO weights, validation against ordinary kriging, exceedance maps |
| [05_08_02_bayesian_kriging.ipynb](05_08_02_bayesian_kriging.ipynb) | `BayesianKriging` | Grid posterior over range and nugget, priors, model choice, the Student-t mixture |
| [05_08_03_bayesian_glm_kriging.ipynb](05_08_03_bayesian_glm_kriging.ipynb) | `BayesianGLMKriging` | Poisson lung-cancer counts and binomial diabetes prevalence, with MCMC |
| [05_08_04_bayesian_transformed_kriging.ipynb](05_08_04_bayesian_transformed_kriging.ipynb) | `BayesianTransformedKriging` | Box-Cox λ on the grid, back-transformed draws, Jura and Meuse metals |
| [05_08_05_conjugate_bayesian_kriging.ipynb](05_08_05_conjugate_bayesian_kriging.ipynb) | `ConjugateBayesianKriging` | Fixed correlation, the Student-t predictive, `from_variogram`, trend priors |
| [05_08_06_bayesian_kriging_function.ipynb](05_08_06_bayesian_kriging_function.ipynb) | `bayesian_kriging` | One call that returns mean, standard deviation and quantiles |
| [05_08_07_correlation_function.ipynb](05_08_07_correlation_function.ipynb) | `correlation_function` | The six correlation models, practical range, Matérn smoothness, the stable family |


---

## Types of Bayesian kriging

### Conjugate Bayesian kriging (known correlation)

If the correlation function is fixed and only β and σ² are unknown, a normal-inverse-gamma prior is conjugate. The predictive distribution is then a Student-t with a closed form (Kitanidis, 1986; Handcock & Stein, 1993). The mean looks like universal kriging, and the variance is inflated to account for the uncertain trend and sill. It is fast and exact, but leaves the range and nugget fixed, which is where most of the uncertainty usually sits.

GeoLattice: `ConjugateBayesianKriging`, or `BayesianKriging.from_variogram` when the fixed correlation comes from a fitted variogram. Tutorial: [conjugate Bayesian kriging](05_08_05_conjugate_bayesian_kriging.ipynb).

### Bayesian kriging with a discrete posterior (model-based geostatistics)

Here the range and the relative nugget get priors as well. GeoLattice puts them on a finite grid and, at every grid point, integrates β and σ² out in closed form (Kitanidis, 1986; Handcock & Stein, 1993; Diggle & Ribeiro, 2007, the `krige.bayes` approach in geoR). The predictive distribution is a finite mixture of Student-t distributions, so the mean, variance, quantiles and exceedance probabilities are exact. Every prediction then reflects uncertainty in the range and nugget too. The price is one covariance factorization per grid point, which suits a few thousand observations.

GeoLattice: `BayesianKriging`, and the one-call wrapper `bayesian_kriging`. Tutorials: [Bayesian kriging](05_08_02_bayesian_kriging.ipynb) and [the one-call function](05_08_06_bayesian_kriging_function.ipynb).

### Bayesian transformed Gaussian kriging

Many variables are skewed: concentrations, rainfall, grades. De Oliveira, Kedem & Short (1997) put a prior on the Box-Cox transformation parameter as well, so the choice of transformation is part of the uncertainty rather than a decision made beforehand. Back-transformed predictions come from posterior draws on the data scale, which carries the median, the mean, the quantiles and the exceedance probabilities in one set of samples.

GeoLattice: `BayesianTransformedKriging`. Tutorial: [Bayesian transformed kriging](05_08_04_bayesian_transformed_kriging.ipynb).

### Bayesian generalized kriging (spatial GLMMs)

For counts, presence/absence or proportions, the Gaussian field sits inside a generalized linear model. A Poisson count has a log mean equal to a trend plus a Gaussian random field; a binomial observation uses a logit link. The posterior has no closed form, so GeoLattice samples it by MCMC (Diggle, Tawn & Moyeed, 1998; Christensen, 2004). Disease mapping and species distribution models are typical uses.

GeoLattice: `BayesianGLMKriging`, families `"poisson"` and `"binomial"`. Tutorial: [Bayesian GLM kriging](05_08_03_bayesian_glm_kriging.ipynb).

### Empirical Bayesian kriging (EBK)

EBK (Krivoruchko, 2012) accounts for variogram uncertainty without a prior and without MCMC. GeoLattice implements the global form. It fits one variogram to the data, simulates new values at the same locations, refits a variogram to each simulation, and weights those variograms by how well each one leave-one-out predicts its own simulation. Predictions are a weighted mixture of ordinary or simple kriging over that ensemble. The prediction variance splits into the usual kriging variance and a between-variogram term, which is the variogram uncertainty plug-in kriging leaves out. The predictive distribution is a mixture of Gaussians, so exceedance probabilities are available directly.

GeoLattice: `EmpiricalBayesianKriging` in `geolattice.geostatistics.kriging.ebk`. Tutorial: [empirical Bayesian kriging](05_08_01_empirical_bayesian_kriging.ipynb).

### The correlation function

Every class above builds its covariance from a correlation ρ(h) = 1 − γ(h), with γ a unit-sill, zero-nugget variogram. `correlation_function` evaluates that correlation for the six models GeoLattice supports. The Bayesian classes call it internally; it is also the right tool for building a covariance matrix or for comparing models before a fit.

GeoLattice: `correlation_function`. Tutorial: [correlation functions](05_08_07_correlation_function.ipynb).


---

## Bayesian kriging in GeoLattice

The classes share a fit-then-predict pattern. Coordinates are arrays of shape `(n, 2)`, used as given. For geometric anisotropy, transform them first with `geolattice.geostatistics.transform_aniso`. Continuous-data models write

$$
Z(\mathbf{x}) = \mathbf{f}(\mathbf{x})^\top\boldsymbol\beta + S(\mathbf{x}) + \varepsilon(\mathbf{x}),
\qquad
\mathrm{Cov}(Z) = \sigma^2\bigl(\mathbf{R}(\phi) + \tau\mathbf{I}\bigr),
$$

where $\mathbf{R}(\phi)$ is the correlation matrix from `correlation_function`, $\phi$ is the range in the `Variogram` convention of the chosen model, and $\tau$ is the nugget relative to the partial sill. The trend $\mathbf{f}$ is a polynomial in the coordinates (`trend` 0 constant, 1 linear, 2 quadratic) plus optional covariates.

### `correlation_function`

```python
from geolattice.geostatistics import correlation_function

rho = correlation_function(h, model="exponential", range_=1.0, nu=1.5, alpha=1.0)
```

`rho` is ρ(h), with ρ(0) = 1. `model` is `"spherical"`, `"exponential"`, `"gaussian"`, `"matern"`, `"cubic"` or `"stable"`. `range_` follows the `Variogram` convention of that model, so a fitted variogram's range can be passed straight through. `nu` is the Matérn smoothness (default 1.5; ν = 0.5 is the exponential). `alpha` is the stable exponent in (0, 2] (1 is exponential, 2 is Gaussian). In the Bayesian classes the same number is passed as `nu` for both the Matérn smoothness and the stable exponent.

### `EmpiricalBayesianKriging`

Module `geolattice.geostatistics.kriging.ebk`. An ensemble of ordinary or simple kriging models. Each member kriges the real data with its own refitted variogram. `predict` returns the weighted mean. `predict_proba` evaluates the mixture of Gaussians at one or more cutoffs.

```python
from geolattice.geostatistics import EmpiricalBayesianKriging

ebk = EmpiricalBayesianKriging(
    model="spherical",
    kriging_type="ordinary",   # or "simple"
    n_realizations=100,
    use_gpu="auto",
).fit(coords, values)
z_hat, se = ebk.predict(grid, return_variance=True)
prob = ebk.predict_proba(grid, cutoffs=[0.8])
```

After `fit`, `base_variogram_` is the variogram of the real data (used only to drive the simulations), `variograms_` holds the refits, `weights_` the inverse leave-one-out MSE weights, and `krige_models_` the kriging models that `predict` actually uses. With `return_variance=True`, `predict` returns that standard error as the square root of two variances: the weighted kriging variance of the ensemble members, and the weighted variance of their means. Pass `variogram_kwargs` (for example `n_lags`, `estimator`, `maxlag`) through to every `Variogram` fit. The simulate-and-refit loop runs on CPU; `use_gpu` applies when the ensemble predicts. `predict_proba` returns a dictionary, one probability array per cutoff.

### `ConjugateBayesianKriging`

Fixed range and relative nugget. Unknown trend and variance, with a conjugate normal-inverse-gamma prior and a Student-t predictive.

```python
from geolattice.geostatistics import BayesianKriging, ConjugateBayesianKriging

ck = ConjugateBayesianKriging(
    "spherical", range_=1.2, nugget_ratio=0.2, trend=0,
).fit(coords, values)

# the same model, built from a fitted Variogram
ck = BayesianKriging.from_variogram(variogram, trend=0).fit(coords, values)

mean, sd = ck.predict(grid, return_std=True, signal=True)
```

`beta_prior` is `"flat"` or `("normal", mean, cov)`, with `cov` scaled by σ². `sigma2_prior` is `"reciprocal"` or `("scaled_inv_chi2", df, scale2)`. `signal=True` predicts the noise-free process. `signal=False` predicts a new observation, nugget included, which is the right target for a held-out sample. `predict_quantiles`, `predict_proba` and `predictive_samples` (with `joint=True` for a joint draw) use the same Student-t.

### `BayesianKriging`

The range and the relative nugget sit on discrete grids. For every pair the conjugate prior integrates β and σ² out, and the marginal likelihood supplies the posterior weight. The predictive distribution is the corresponding mixture of Student-t distributions.

```python
bk = BayesianKriging(
    model="exponential",
    trend=0,
    range_grid=None,                  # default: 30 values, log-spaced from 2% to 100% of the largest distance
    nugget_grid=(0.0, 0.1, 0.25, 0.5, 1.0),
    range_prior="uniform",            # or "reciprocal", "squared_reciprocal", or an array of weights
    nugget_prior="uniform",
    beta_prior="flat",
    sigma2_prior="reciprocal",
).fit(coords, values, covariates=None)

mean, sd = bk.predict(grid, return_std=True, signal=True)
quantiles = bk.predict_quantiles(grid, quantiles=(0.05, 0.5, 0.95))
exceedance = bk.predict_proba(grid, threshold=0.8)
draws = bk.sample_parameters(n=1000)
```

`posterior_` is a table with one row per grid point: `range`, `nugget_ratio`, `log_prior`, `log_marginal`, `weight`, `dof`, `sigma2_mean`, and one `beta_*` column per trend coefficient. `nu` is the Matérn smoothness or the stable exponent, depending on `model`.

### `BayesianTransformedKriging`

`BayesianKriging` with a Box-Cox parameter λ on the same grid. The data must be strictly positive. The transform is the geometric-mean normalised Box-Cox, so marginal likelihoods are comparable across λ. Predictions are Monte Carlo draws, inverted back to the original units.

```python
from geolattice.geostatistics import BayesianTransformedKriging

bt = BayesianTransformedKriging(
    "exponential",
    lambda_grid=(-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0),
    lambda_prior="uniform",
).fit(coords, values)

table = bt.predict(grid, n_samples=2000)   # columns: mean, median, sd, q05, q95
```

`lambda_posterior_` collects the posterior weight of each λ. `predict_quantiles` and `predict_proba` summarise the same back-transformed draws. For λ < 0 some draws fall outside the Box-Cox domain and are infinite, so the predictive mean need not exist; the median and the quantiles remain the summaries to report. `posterior_` gains a `lambda` column.

### `BayesianGLMKriging`

A spatial generalized linear mixed model, sampled by MCMC.

- `"poisson"`: $y_i \mid \eta_i \sim \mathrm{Poisson}(E_i e^{\eta_i})$, with exposure or expected count $E_i$.
- `"binomial"`: $y_i \mid \eta_i \sim \mathrm{Binomial}(n_i, \mathrm{logit}^{-1}(\eta_i))$, with $n_i$ trials.

The linear predictor is a trend plus a latent Gaussian field. The sampler whitens that field and updates it with MALA, updates β and log σ² by random-walk Metropolis, and moves the range on a discrete grid. With `interweave=True` (the default) each iteration also updates β, σ² and the range in the centred parametrization with the linear predictor held fixed, which mixes the intercept, the variance and the range.

```python
from geolattice.geostatistics import BayesianGLMKriging

glm = BayesianGLMKriging(
    family="poisson",          # or "binomial"
    model="exponential",
    trend=0,
    nugget_ratio=0.0,
    sigma2_prior=(1.0, 0.5),   # scaled-inv-chi2 (df, scale2)
    beta_prior_sd=10.0,
    interweave=True,
).fit(coords, counts, trials_or_exposure=expected, n_iter=6000, burn=2000, thin=4)

table = glm.predict(grid, scale="response")   # mean, sd, median, q05, q95
rate_exceeds = glm.predict_proba(grid, threshold=1.0, scale="response")
```

`trials_or_exposure` is the expected count or exposure for Poisson (default 1) and the number of trials for binomial (default 1). `scale="link"` returns the linear predictor; `"response"` returns the rate or the probability. `trace_` holds the kept draws of β, σ², range and log-likelihood. `acceptance_` holds the acceptance rates after burn-in.

### `bayesian_kriging`

One call to `BayesianKriging`: fit, then a table of the posterior predictive mean, standard deviation and quantiles.

```python
from geolattice.geostatistics import bayesian_kriging

table = bayesian_kriging(coords, values, grid, model="exponential", trend=0)
# columns: mean, sd, q05, q50, q95

table, model = bayesian_kriging(
    coords, values, grid,
    model="matern", nu=1.5,
    quantiles=(0.05, 0.5, 0.95),
    signal=True,
    return_model=True,
    range_grid=None,
    nugget_grid=(0.0, 0.1, 0.25, 0.5, 1.0),
)
```

`signal=False` predicts a new observation, including the nugget. `return_model=True` also returns the fitted `BayesianKriging`, so `posterior_`, `predict_proba` and `sample_parameters` stay available. Keyword arguments are the `BayesianKriging` arguments (`range_grid`, `nugget_grid`, `range_prior`, `nugget_prior`, `beta_prior`, `sigma2_prior`, `nu`). Covariates go in `covariates` at the data and `covariates_pred` at the prediction locations.


---

## Choosing among them

| Situation | Use | Notebook |
|---|---|---|
| The correlation model is already chosen; the trend and the sill are still uncertain | `ConjugateBayesianKriging`, or `BayesianKriging.from_variogram` | [05](05_08_05_conjugate_bayesian_kriging.ipynb) |
| Range and nugget are uncertain, the data are continuous, and there are up to a few thousand points | `BayesianKriging` | [02](05_08_02_bayesian_kriging.ipynb) |
| The same model, as a table of predictions in one call | `bayesian_kriging` | [06](05_08_06_bayesian_kriging_function.ipynb) |
| Positive, skewed measurements, and the transform itself is uncertain | `BayesianTransformedKriging` | [04](05_08_04_bayesian_transformed_kriging.ipynb) |
| Counts (Poisson, with exposure) or trials (binomial) | `BayesianGLMKriging` | [03](05_08_03_bayesian_glm_kriging.ipynb) |
| Routine mapping of a continuous variable, where variogram uncertainty matters and a prior or an MCMC run is more than the job needs | `EmpiricalBayesianKriging` | [01](05_08_01_empirical_bayesian_kriging.ipynb) |
| A covariance matrix, a simulated field, or a comparison of correlation models | `correlation_function` | [07](05_08_07_correlation_function.ipynb) |


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.